In [1]:
import pandas as pd

from src import ProfileReport, Settings
from src import ScoringAgent, TimeSeriesAgent, MetaDataAgent

/usr/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATASET = "./data/Steep_cleaned_cleaned.csv"
df = pd.read_csv(DATASET)

In [3]:
cols = df.columns.to_list()

# Check if we're dealing with timeseries
ts_agent = TimeSeriesAgent()
ts_mode, ts_col = ts_agent.is_timeserie(cols)

if ts_mode:
    df[ts_col] = pd.to_datetime(df[ts_col])


# Sampling
df = df.sample(frac=0.01, random_state=42)

{"ts_mode": true, "timestamp_col": "Timestamp"}


In [4]:
meta_agent = MetaDataAgent()
meta_cols = meta_agent.get_metadata(cols)

{'Timestamp': 'The date and time when the data was recorded.', 'Temps de cycle (s)': 'The total cycle time in seconds for the process.', "Temps d'injection (s)": 'The duration of the injection phase in seconds.', 'Matelas matière (ccm)': 'The material cushion volume in cubic centimeters.', 'Temps de dosage (s)': 'The time taken for dosing in seconds.', 'Commutation postpression Pression injection (bar_spéc)': 'The injection pressure during post-pressure switching, measured in bar.', 'Pression injection Integrale (bar_spez*s)': 'The integral of the injection pressure over time, measured in bar-seconds.', 'Position vis à fin mantien en pression (ccm)': 'The screw position at the end of pressure holding, measured in cubic centimeters.', 'Commutation postpression Position de vis (ccm)': 'The screw position during post-pressure switching, measured in cubic centimeters.', 'BU1 Temp.réel (°C)': 'The real-time temperature of BU1 in degrees Celsius.', 'Pression injection max (bar_spéc)': 'The m

In [6]:
settings = Settings()
settings.html.inline = False  # separate assets instead of inline
settings.html.style.logo = "assets/logo.png"


report = ProfileReport(df,
                       title="Obelix Data Quality Report",
                       progress_bar=True,
                       #
                       explorative=True,
                       minimal=False,
                       sensitive=True,
                       #
                       sortby=ts_col,
                       sort="ascending" if ts_mode else None,
                       tsmode=ts_mode,
                       #
                       variables=meta_cols,
                       #
                       config=settings)

report.to_file("output/report.html")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 112.91it/s]


In [7]:
# scoring
import json
report_json = report.to_json()
report_json = json.loads(report_json)

Render JSON: 100%|██████████| 1/1 [00:00<00:00,  1.56it/s]


In [8]:
scr_agent = ScoringAgent()
score, suggs = scr_agent.get_score(report_json['alerts'])

print(score)
for _ in suggs:
    print("* " + _)

{'score': 65, 'suggs': ["One of your fields has a constant value. This means it doesn't change and provides no useful information. Check the source of this data to ensure it's being recorded correctly.", 'Several fields are highly correlated, meaning they provide similar information. Consider removing redundant fields to simplify analysis and avoid confusion.', 'Some fields are highly imbalanced. This means certain values dominate, which can lead to biased results. Try collecting more diverse data to balance these fields.', 'Many fields are non-stationary, meaning their patterns change over time. This could impact time-based analysis. Review the processes generating this data to see if there are predictable cycles or trends.', "One field is seasonal, meaning it changes in a regular pattern over time. This might be normal but ensure you're accounting for this pattern in analysis.", 'Ensure timestamps are accurate and properly synchronized with other data sources. This is critical for an